In [ ]:
# Locate the project root (where config.yaml lives) and set as cwd
import os
from pathlib import Path
for _parent in [Path.cwd(), Path.cwd().parent]:
    if (_parent / 'config.yaml').exists():
        os.chdir(_parent)
        break
print('Working directory:', os.getcwd())


# Step 06: Visualization


Generates MA plots (shrunk and unshrunk), volcano plots, and PCA for **all contrasts** from Step 05. Outputs to `Figures/06_visualization/`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from scripts.visualization import ma_plot, volcano_plot, pca_plot
from utils.helpers import ensure_dir
import pandas as pd

counts = pd.read_csv('Data/count_matrix_processed.tsv', sep='\t', index_col=0)
sheet = pd.read_csv('Data/samplesheet.csv').set_index('sample')
meta = sheet.loc[counts.index].rename(columns={'group': 'condition'})

out_dir = 'Figures/06_visualization'
ensure_dir(out_dir)

# All contrasts from config.yaml
contrasts = ['siKRAS_vs_siScr', 'siKRAS_vs_Empty', 'siKRAS_vs_Control', 'Empty_vs_Control']

for label in contrasts:
    # Shrunk log2FC
    res = pd.read_csv(f'Data/tables/deseq2_results_{label}.tsv', sep='\t', index_col=0)
    ma_plot(res, f'{out_dir}/{label}_MA_plot_shrunk.png', shrunken=True)
    volcano_plot(res, f'{out_dir}/{label}_volcano_plot.png')
    # Unshrunk log2FC (for comparison)
    res_un = pd.read_csv(f'Data/tables/deseq2_results_unshrunk_{label}.tsv', sep='\t', index_col=0)
    ma_plot(res_un, f'{out_dir}/{label}_MA_plot_unshrunk.png')

# PCA (independent of contrast)
pca_plot(counts, meta, f'{out_dir}/PCA_plot.png')
